# Lab 4.3 &mdash; Prompt, Fine-tune, or RAG? See Why Fine-tuning Wins Here

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 15 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Tell prompting, fine-tuning and RAG apart
- See a fine-tuned model succeed where a general model fails
- Build the judgement you'll use on every real project

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Prompt vs fine-tune vs RAG](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-03")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Three ways to adapt a pre-trained model:
- **Prompt** &mdash; just ask, no training. Fast, flexible, good default.
- **Fine-tune** &mdash; train further on your labelled data. Best for a fixed, specialised task/style (this module).
- **RAG** &mdash; fetch your documents at query time. Best when answers must come from *your* up-to-date knowledge (Day 3).

To *feel* why fine-tuning matters, we compare a **fine-tuned** sentiment model against a **general**
model that was never fine-tuned for sentiment &mdash; on the same sentences.

## Build it
Pick the two real models: one fine-tuned for sentiment, one general (has a random sentiment head).

In [ ]:
try:
    FINETUNED = GENERAL = None   # filled in below
    FINETUNED = ___   # TODO: "distilbert-base-uncased-finetuned-sst-2-english"
    GENERAL   = ___   # TODO: "prajjwal1/bert-tiny"  (never fine-tuned for sentiment)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## Run it for real
Run both real models on the same sentences and compare.

In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    SENTS = ["a brilliant and moving masterpiece", "a boring dreadful waste of time", "i loved it truly excellent"]
    TRUE  = [1, 0, 1]

    def run_model(name):
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2); mdl.eval()
        enc = tok(SENTS, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad(): pred = mdl(**enc).logits.argmax(-1).tolist()
        return pred

    ft = run_model(FINETUNED)
    gen = run_model(GENERAL)
    print("truth            :", TRUE)
    print("fine-tuned model :", ft, " correct:", sum(p==t for p,t in zip(ft, TRUE)), "/3")
    print("general model    :", gen, " correct:", sum(p==t for p,t in zip(gen, TRUE)), "/3", "  <- random head, no idea")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The **fine-tuned** model gets sentiment right; the **general** bert-tiny (its sentiment head is randomly initialised) is essentially guessing. **Fine-tuning is what specialised the first one.**
- If a plain prompt to a general model already solved your task, you would **prompt** and skip training. When it cannot &mdash; as here &mdash; you **fine-tune** (labs 4&ndash;12).
- When the answer must come from *your own changing documents*, neither helps: you reach for **RAG** (Day 3).

## Your turn (open task &mdash; no grader)
Map each scenario to `"prompt"`, `"fine-tune"`, or `"rag"` and justify it: (a) draft a one-off
tagline; (b) classify 50k tickets into your 8 fixed categories; (c) answer staff questions from
internal policy PDFs; (d) always reply in your brand voice. No grader &mdash; a "good" answer explains
*why*, and names which one you would try **first** and how you would know it was enough.

---
### Key takeaway
Default to **prompting**; **fine-tune** for a fixed specialised task (this module); reach for **RAG** when answers must come from your own changing data (Day 3).

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>